# Lab Assignment: Generative AI

#### Lab team: 18

##### Name (member 1): Miguel Borregón Sánchez

##### Name (member 2): Belisa Yu

* Please include your full name at the beginning of all submitted files.
* Make sure the presentation is well-structured: the report will be evaluated not only for correctness, but also for clarity, conciseness, and completeness.
* Make use of figures and tables to summarize the results and illustrate the discussions.
* If external material is used, the sources must be cited. 
* Include references in APA format https://pitt.libguides.com/citationhelp/apa7. Lack or poorly formatted references can be penalized.
* A generative AI tool can be used for consultation. You must specify the tool used in your report.
* You are not allowed to use a generative AI tool to generate code.

Submit a single `.zip` file, whose name has the format `AA3_2024_2025_P02_teamCode_lastName1_lastName2.zip` 
The name must not include graphical accents, spaces, uppercase letters, or special characters.

For example: `AA3_2024_2025_P02_V03_munyoz_deLaRosa.zip`

This compressed file must include the following files:

* This Python notebook with the solutions of the exercises. The notebook should include only code snippets, figures, tables, derviations and explanations (with LaTex if necessary) in Markdown cells. Handwritten material can be included in the Python notebook as images. Functions should be defined in a separate `.py` file, not in the notebook.
* The necessary `.py` and addional files to ensure the Python notebook code can be executed sequentially without errors.
* A PDF file generated from the notebook (Export the notebook as an HTML file. Open the HTML file in a Browser and print it as a PDF file). 

Make sure that all the code cells can be executed squentially without errors (Kernel -> Restart & Run All). Exectution and formatting errors will be penalized.

The grade of this lab assignment is based on
* This submission (50 %).
* An individual in-class exam (50%).


Evaluation criteria:
* [6 points] Quality of the report (correctness, clarity, conciseness, completeness).
* [3 points] Quality of the code (correctness, adherence to a Python style guide -for instance, Google's-, comments, functional decomposition).
* [1 point]  References.                                                                   

## References:

1. Yang Song, Jascha Sohl-Dickstein, Diederik P. Kingma, Abhishek Kumar, Stefano Ermon and Ben Poole
"Score-Based Generative Modeling through Stochastic Differential Equations"
In International Conference on Learning Representations, 2021
https://arxiv.org/abs/2011.13456
2. TODO: Include references in alphabetical order

## Exercise 1:  Generative AI based on diffusion models: Brownian motion 

In this exercise the training data is generated by injectiong noise using Brownian motion (variance exploding diffusion model).
The stochastic differencial equation (SDE) that characterizes this forward diffusion process is
$$
d\mathbf{x}(t) = \sigma^t d\mathbf{W}(t) 
$$

with $ \mathbf{x}(0) \sim p_0\left(\mathbf{x} \right)$, and  $\sigma > 0 $.
1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary t?
2. What is the SDE for the time-reversed process? 
3. For the synthesis of new images
    1. In what interval is the reverse process SDE integrated?
    2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled?

### Exercise 1.1:  Training the model by weighted sum of denoising score matching objectives
1. Give the mathematical form and explain the cost function used in the training of the generative AI model. 

$$
\min_{\theta} \mathbb{E}_{t \sim \mathcal{U}[0,T]} \left[ \lambda(t) \; \mathbb{E}_{\mathbf{x}(0) \sim p_{\text{data}}(\mathbf{x})} \; \mathbb{E}_{\mathbf{x}(t) \sim p_{0t}(\mathbf{x}(t) \mid \mathbf{x}(0))} \left[ \| \mathbf{s}_\theta(\mathbf{x}(t), t) - \nabla_{\mathbf{x}(t)} \log p_{0t}(\mathbf{x}(t) \mid \mathbf{x}(0)) \|^2 \right] \right]
$$

Pero para ruido gaussiano se simplifica y obtenemos la siguiente función de coste:
$$
\min_{\theta} \mathbb{E}_{t, \mathbf{x}_0, \mathbf{z}} \left[ \lambda(t) \left\| \mathbf{s}_\theta(\mathbf{x}_0 + \sigma_t \mathbf{z}, t) + \frac{\mathbf{z}}{\sigma_t} \right\|^2 \right]
$$
Esto es debido a que cuando añadimos ruido gaussiano, la distribución es: $x_t = x_0 + \sigma_t z$ donde z~N(0,I), lo que implica que $x_t | x_0 ~ N(x_0, \sigma^2_t I)$

Para una distribución normal, el logaritmo de la densidad es: $log \ p(x) = -\frac{(x-mu)^2}{2\sigma^2} + constante$

Por tanto: $\nabla_{x_{t}} log \ p_0t (x_t|x_0) = -\frac{\sigma_t z}{\sigma^2_t} = - \frac{z}{\sigma_t}$

Sustituyendo en la fórmula general: $\left\| \mathbf{s}_\theta(\mathbf{x}_t, t) - \left(-\frac{\mathbf{z}}{\sigma_t}\right) \right\|^2 = \left\| \mathbf{s}_\theta(\mathbf{x}_t, t) + \frac{\mathbf{z}}{\sigma_t} \right\|^2$

2. Indicate what type of neural network is used to model the score, its inputs, outputs, and architecture.   

Se usa una red U-Net (usado mucho en procesamiento de imágenes, sobre todo médicas), cuyo funcionamiento interno consiste en analizar la imagen de forma global para capturar el contexto, y luego ir evaluando sus características a diferentes escalas (dividiendo la imagen en 'trozos'), para luego recomponerla y obtener el resultado esperado<br>
- Inputs: 
    - imagen con ruido
    - instante de tiempo t (positional encoding)
- Output: Score $\nabla \ log \ p_t$ -> tensor de la misma dimensión que la entrada

*anexo*: Aunque la U-Net sea muy potente, cuando la imagen de entrada tiene mucho ruido, es prácticamente imposible que la red recupere una imagen limpia en un solo paso, ya que existen infinitas imágenes posibles compatibles con ese ruido. Si forzamos una sola pasada, el resultado sería borroso y con mucho ruido residual. Por eso, en los diffusion models se utiliza un proceso iterativo: comenzamos con ruido puro y aplicamos la U-Net muchas veces (por ejemplo, 1000 pasos), donde cada vez la imagen de entrada tiene un poco menos de ruido que la anterior. De esta forma, en lugar de pedirle algo mágico (la imagen 'buena' en una sola pasada), le pedimos muchos pequeños refinamientos sucesivos, obteniendo finalmente imágenes nítidas y de alta calidad.

3. Explain how time is input in the neural network model for the time-dependent score function using random Fourier features.

Random fourier features permite que una red neuronal simple imite el comportamiento de métodos kernel más sofistificados cuando hay suficientes neuronas; convierte un simple número (el tiempo t) en un montón de números diferentes que ayudan a la red a entender mejor ese tiempo. 

4. Illustrate how to train a model to generate handwritten digits for the MNIST dataset using the Brownian motion diffusion model.
This first block of code should run in less than 5 minutes.

#### esto de aqui habra que quitarlo pero de momento... Referencias:
- https://colab.research.google.com/drive/120kYYBOVa1i0TD85RjlEkFjaWDxSFUx3?usp=sharing#scrollTo=XCR6m0HjWGVV
- https://es.wikipedia.org/wiki/U-Net

### Exercise 1.2: Generation using the Euler-Maruyama integrator
Use a model that you have previously trained (not the one in the previous exercise) high-quality model to generate some samples using the Euler-Maruyama integrator.

### Exercise 1.3: Generation using a probability flow ODE.

(código hecho)

1. Explain what is the Fokker-Planck equation and in what way is it related to an SDE.
Es una ecuación diferencial parcial que describe la evolución temporal de la función de densidad de probabilidad p(x,t) de un proceso estocástico.

Ambas describen el proceso estocástico pero desde perspectivas diferentes:
- SDE: describe el comportamiento de una sola trayectoria
- Fokker-Planck: describe la evolución de la densidad de probabilidad.

2. Explain how the probability flow ODE can be used to generate samples from  $p_0\left(\mathbf{x} \right)$.
Transformando la SDE reversa en una ecuación diferencial determinista. Consiste en partir de la imagen con ruido t = T y llegar a la imagen realista (sin ruido) t = 0, sin necesidad de añadir ruido en cada paso.

3. Implement and illustrate the use of this method to generate synthetic images of handwritten digits. 
ODE.py

4. Indicate how to use this scheme to compute likelihoods. Implement this functionality and illustrate its use.
Algoritmo para calcular la verosimilitud de una imagen:
- Partir de la imagen limpia $x_0$ en t = 0
- Integrar hacia adelante en el tiempo resolviendo la ODE:
$$
\frac{dx}{dt} = f(x,t) - \frac{1}{2} f(t)^{2} \nabla log \ p_t(x)
$$
- Acumular la divergencia a lo largo de la trayectoria:
$$
log \ p_0(x_0) = log \ p_T(x_T) + \int_{0}^{T} \nabla \cdot v(x(t),t) dt
$$
- Dado p_T conocida: en t = T, la distribución es una Gaussiana conocida:
$$
\log p_T(\mathbf{x}_T) = -\frac{\|\mathbf{x}_T\|^2}{2\sigma_T^2} - \frac{D}{2} \log(2\pi\sigma_T^2)
$$

Para el paso 3 (acumular la divergencia a lo largo de la trayectoria), se hará uso de la aproximación de Hutchinson, que se trata de una técnica para estimar la traza de una matriz sin tener que calcular explícitamente todos sus elementos.
$$
E[div_{Hutch}] = \nabla \cdot v
$$

Idea general de Hutchinson (estimación de la traza):
- Generar un vector aleatorio
- Multiplciar ese vector por la matriz
- Multiplicar el resultado por el mismo número
- Obtenemos valor 

*Al repetir esto varias veces con diferentes vectores aleatorios y promediamos, nos acercamos mucho al valor real*

## Exercise 2:  Generative AI based on diffusion models: The Ornstein-Uhlenbeck process

In this exercise, the training data is generated by injecting noise using Brownian motion (variance preserving diffusion model).
The stochastic differencial equation (SDE) that characterizes this forward diffusion process is
$$
d\mathbf{x}(t) = - \frac{1}{2} \beta(t) \mathbf{x}(t) + \sqrt{\beta(t)} d\mathbf{W}(t) 
$$
with $ \mathbf{x}(0) \sim p_0\left(\mathbf{x} \right)$
1. What is the distribution of $\mathbf{x}(t)$ assuming that $\mathbf{x}(0) = \mathbf{x}_0$ for arbitrary t?
2. What is the SDE for the time-reversed process? 
3. For the synthesis of new images
        1. In what interval is the reverse process SDE integrated?
        2. From which distribution $\pi\left(\mathbf{x} \right)$ is the initial condition of the reverse SDE sampled? 

#### **Soluciones**
1. Ecuación de la SDE lineal: $dx = \alpha(t)x dt + \gamma(t) dW$. En este caso:
- $\alpha(t) = - \frac{1}{2}\beta(t)$
- $\gamma(t) = \sqrt{\beta(t)}$
Aplicamos el método del factor integrante:

$$
\Phi(t) = \exp\left( -\int_0^t \alpha(s) \, ds \right) = \exp\left( \frac{1}{2} \int_0^t \beta(s) \, ds \right)
$$

Donde:
- $\bar{\beta} = \int_{0}^{t}\beta(s)ds$, entonces $\Phi(t) = \exp{\frac{1}{2}\bar{\beta}(t)}$
- $d\bar{\beta}(t) = \beta(t)dt$

Sea $y(t) = \Phi(t)x(t)$. Aplicando el lema de Itô: $dy = \Phi dx + x d\Phi$:
- $d\Phi = \frac{d\Phi}{d\bar{\beta}}\cdot{d\bar{\beta}} \ = \frac{1}{2}\Phi(t)\beta(t)$
- Sustituyendo en el lema de Itô:
$$
\Phi(-\frac{1}{2}\beta x dt + \sqrt{\beta}dW) + x (\frac{1}{2}\Phi \beta dt) \ = \ \Phi(t)\sqrt{\beta(t)}dW(t)
$$

Integramos desde 0 a t: $\int_{0}^{t}dy(s) = \int_{0}^{t} \Phi(s)\sqrt{\beta}(s)dW$
- Teorema del cálculo estocástico: $\int_{0}^{t}dy(s) = y(t)-y(0)$
- y(t) - y(0) = $\int_{0}^{t}\Phi(s)\sqrt{\beta(s)}dW(s)$
- y(0) = $\Phi(0)x(0) = \exp(\frac{1}{2}\int_{0}^{0}\beta(s)ds) = \exp(0) = 1$, por tanto, y(0) = $x_0$
- $\Phi(t)x(t) - x_0  = \int_{0}^{t} \Phi(s)\sqrt{\beta(s)}dW(s)$
- $x(t) = \Phi(t)^{-1}x_0 + \Phi(t)^{-1} \int_{0}^{t}\Phi(s)\sqrt{\beta(s)}dW(s)$
- $\Phi(t)^{-1} = \exp{-\frac{1}{2}\bar{\beta}(t)}$
- $x(t) =\exp^{-\frac{1}{2}\bar{\beta}(t)} x_0 + \exp^{-\frac{1}{2}\bar{\beta}(t)}\int_{0}^{t}\Phi(s)\sqrt{\beta(s)}dW(s)$
- Ahora operaremos paso a paso:
    - Resolver la integral $\int_{0}^{t} (\exp^{\frac{1}{2}\bar{\beta}(s)}\sqrt{\beta}(s)) ds$ = $\int_{0}^{t} (\exp^{\frac{1}{2}\bar{\beta}(s)}\sqrt{\beta}(s))^{2}ds$ = $\int_{0}^{t} (\exp^{\bar{\beta}(s)}\beta(s))$. <br>
    Dado que $\frac{d}{ds} \exp^{\bar{\beta}(s)} = \exp^{\bar{\beta}(s)}\beta(s)$. La integral es:
    $$
    \int_{0}^{t} (\exp^{\bar{\beta}(s)}\beta(s)) = [\exp^{\bar{\beta}(s)}]_{0}^{t} = \exp^{\bar{\beta}(t)} - \exp^{0} = \exp^{\bar{\beta}(t)} - 1
    $$

    - resultado final:
    $$
    x(t) =\exp^{-\frac{1}{2}\bar{\beta}(t)} x_0 + \exp^{-\frac{1}{2}\bar{\beta}(t)}\cdot (\exp^{\bar{\beta}(t)} - 1)
    $$

2. SDE reversa. Fórmula general: $dx = [f(x,t)-g(t)^{2}\nabla_x log p_t (x)] dt + g(t) d \bar{W}$

En este caso: 
- $\alpha(t) = - \frac{1}{2}\beta(t)$
- $\gamma(t) = \sqrt{\beta(t)}$

Sustiyendo en la fórmula, obtenemos:
$$
dx = [-\frac{1}{2}\beta(t)x - \beta(t)\nabla_x log p_t (x)]dt + \sqrt{\beta(t)}d\bar{W}
$$

3. 
    1. $t \epsilon [T,0]$
    2. La distribución inicial es aproximadamente una N(0,1)

### Exercise 2.1: Training and generation of images using the OU process and different noise schedules
1. Using the linear noise schedule.
2. Using the cosine noise schedule.
3. Using a third noise schedule of your choice.

## Exercise 3:  Evaluation of the quality of the generated images
1. Describe, and compare the characteristics, advantages, and disadvantages of the following measures of quality for generative IA models for images:
    1. The negative log-likelihood (NLL).
    2. Bits per dimension (BPD). 
    3. Fréchet Inception Distance (FID).
    4. A third measure of your choice.
2. Compare using BPD the different diffusion models (Brownian motion, Ornstein-Uhlenbeck), noising schedules (linear, cosine, etc.), and sampling strategies (SDE, ODE) implemented.

#### **Soluciones**
**Ejercicio 1**

- $NNL$ es la log-verosimilitud negativa. Mide cuán de probable es que el modelo genere una imagen real: $NLL = -log \ p(x)$, donde p(x) es la probabilidad que el modelo asigna a la imagen x

- Mide la cantidad de información, en bits, que
necesita el modelo para reconstruir cada dimensión de los datos.
Generalmente se calcula a partir de la pérdida de log-verosimilitud negativa (NNL) o la divergencia KL que produce el modelo. Luego, el resultado se promedia y se divide por 
$log (2)$ para convertir las unidades de nats a bits, lo que facilita la comparación entre diferentes modelos.

- FID compara la distribución de las imágenes generadas con la distribución de un  conjunto
de imágenes de verdad, para ello mide la distancia entre dos distribuciones de probabilidad en el espacio de características de una red neuronal preentrenada (Inception v3).
El FID asume que las distribuciones son Gaussianas
Un valor bajo FID indica que las imágenes generadas son de alta calidad y diversas, ya que su distribución es similar a la de las imágenes reales. Por el contrario, un valor alto sugiere diferencias signifcatobas,
con imágenes borrosas, con artefactos o poca diversidad. Sin embargo, una limitación encontrada, es que se requiere un número considerable de imágenes para estimar correctamente las covarianzas y ausme que las distribuciones
son gaussianas. Además, según demostraron $Chong \ y \ Forsyth$, DIF está sesgado estadísticamente, es decir, su valor esperado sobre datos finitos no es su valor real.

- IS (inception score): El IS se calcula en función de la salida de un modelo de clasificación de imágenes Inception v3 preentrenado
e independiente. Un valor alto indica que las imágenes generadas son nítidas y diversas. Sin embargo, no compara con imágenes reales,
por lo que hoy en día es complementada con FID. Se maximiza cuando cumple las siguientes condiciones:
    - Se minimiza la entropí de la distribución de etiquetas predicha por el modelo Inception v3 para las imágenes generadas. Es decir, predice con confianza una única etiqueta para cada imagen
    - Las predicciones del modelo de clasificación se distribuyen uniformemente entre todas las etiquetas posibles.<br>